# 07 — SHAP Explainability

## Objective

This notebook focuses on interpreting the machine-learning models developed for Landsat-based LST downscaling.

In the previous notebooks, Random Forest (RF) and XGBoost (XGB) were trained and evaluated for predicting 30 m Land Surface Temperature (LST) from spectral and derived predictor variables.

Model comparison showed that RF and XGBoost achieved very similar predictive performance. Therefore, this notebook investigates **why these models make their predictions** using SHAP (SHapley Additive exPlanations).

### Key questions

1. Which predictor variables are most influential in the LST predictions?
2. How does each predictor influence the predicted LST?
3. Do high or low values of a predictor tend to increase or decrease the model's prediction?
4. Are the explanations from Random Forest and XGBoost consistent with each other?

### Why SHAP?

Conventional model performance metrics such as MAE, RMSE, and R² tell us **how well a model predicts LST**, but they do not explain the model's decision-making process.

SHAP provides a way to examine the contribution of individual predictor variables to model predictions. This helps us move from a purely predictive analysis toward an **interpretable machine-learning analysis**.

> **Important:** SHAP describes how the trained model uses the predictor variables. It does not by itself establish physical causation between a predictor and LST.

### Models analyzed

- Random Forest (RF)
- XGBoost (XGB)

The predictor variables and preprocessing used for SHAP analysis will be kept consistent with the corresponding model-training workflows.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Environment ready")

Environment ready


## 7.1 — SHAP and Model Explainability

The Random Forest and XGBoost models developed in the previous notebooks can predict fine-resolution LST, but their predictions do not directly explain which predictors are responsible for those predictions.

SHAP (SHapley Additive exPlanations) is used to interpret the contribution of individual predictor variables to model predictions.

For an individual observation, the SHAP framework expresses the model prediction conceptually as:

\[
f(x) = E[f(X)] + \sum_{j=1}^{p} \phi_j
\]

where:

- \(f(x)\) is the model prediction for a particular observation.
- \(E[f(X)]\) is the average model prediction, also called the **base value** or **expected value**.
- \(p\) is the number of predictor variables.
- \(\phi_j\) is the SHAP value representing the contribution of predictor \(j\).

A positive SHAP value means that the predictor pushes the model prediction higher than the base value, while a negative SHAP value means that it pushes the prediction lower.

SHAP analysis will therefore allow us to investigate both:

1. **Feature importance** — which predictors have the greatest influence on model predictions?
2. **Feature effect** — how do high and low values of each predictor affect the predicted LST?

The analysis will be performed for both Random Forest and XGBoost.

## 7.2 — Import SHAP

The `shap` Python library provides implementations of SHAP-based explainability methods.

For tree-based machine-learning models such as Random Forest and XGBoost, SHAP provides tree-specific explainers that can efficiently calculate feature contributions.

The models used in this project are tree-based models, so a tree-based SHAP explainer is appropriate.

In [2]:
import shap

print("SHAP version:", shap.__version__)

SHAP version: 0.52.0


## 7.3 — Identify the Model Predictors

SHAP explanations are only meaningful when the feature matrix supplied to SHAP is identical to the feature matrix used by the trained model.

Therefore, before calculating SHAP values, the predictor variables and their ordering must be verified from the model-training workflow.

This step prevents an important error: explaining a model using predictors that do not correspond to the variables on which the model was actually trained.

The feature names identified here will be used consistently throughout the SHAP analysis.

In [3]:
# Check the project directories available to Notebook 7

from pathlib import Path

PROJECT_DIR = Path.cwd()

print("Current working directory:")
print(PROJECT_DIR)

print("\nFiles and folders:")
for item in PROJECT_DIR.iterdir():
    print(item.name)

Current working directory:
d:\Projects\GeoAI-LST-Downscaling\notebooks

Files and folders:
01_LST_Retrieval.ipynb
02_Predictor_Generation.ipynb
03_Spatial_Preparation.ipynb
04_RF_Downscaling.ipynb
05_XGB_Downscaling.ipynb
06_Model_Comparison.ipynb
07_SHAP_Interpretation.ipynb


## 7.4 — Set the Project Directory

The SHAP notebook is stored inside the `notebooks` directory, while the project data, results, and figures are stored in directories at the project root.

Therefore, the project root is defined as the parent directory of the notebook's current working directory.

This allows the notebook to access project files using consistent relative paths rather than hard-coded machine-specific paths.

In [4]:
from pathlib import Path

# Current notebook directory
NOTEBOOK_DIR = Path.cwd()

# Project root directory
PROJECT_DIR = NOTEBOOK_DIR.parent

print("Notebook directory:")
print(NOTEBOOK_DIR)

print("\nProject directory:")
print(PROJECT_DIR)

Notebook directory:
d:\Projects\GeoAI-LST-Downscaling\notebooks

Project directory:
d:\Projects\GeoAI-LST-Downscaling


## 7.5 — Inspect Project Data and Results

Before loading the model inputs, the available project directories are inspected.

This helps identify the files generated by the previous modelling notebooks and prevents us from assuming filenames or locations that may not exist.

In [5]:
# Inspect the main project directories

for folder_name in ["data", "results", "figures"]:
    folder = PROJECT_DIR / folder_name

    print(f"\n{'=' * 50}")
    print(f"{folder_name.upper()}")
    print(f"{'=' * 50}")

    if folder.exists():
        for item in sorted(folder.rglob("*")):
            if item.is_file():
                print(item.relative_to(PROJECT_DIR))
    else:
        print("Directory not found.")


DATA
data\processed\lucknow\coarse\lst_90m_lucknow.tif
data\processed\lucknow\downscaled\rf_lst_30m_lucknow.tif
data\processed\lucknow\downscaled\tsharp_lst_30m_lucknow.tif
data\processed\lucknow\downscaled\xgb_lst_30m_lucknow.tif
data\processed\lucknow\fine\indices\fvc_30m_lucknow.tif
data\processed\lucknow\fine\predictors\green_lucknow_reflectance.tif
data\processed\lucknow\fine\predictors\lst_celsius_lucknow.tif
data\processed\lucknow\fine\predictors\lst_clean_lucknow.tif
data\processed\lucknow\fine\predictors\lst_uncertainty_lucknow.tif
data\processed\lucknow\fine\predictors\lucknow_quality_mask.tif
data\processed\lucknow\fine\predictors\mndwi_lucknow.tif
data\processed\lucknow\fine\predictors\ndbi_lucknow.tif
data\processed\lucknow\fine\predictors\ndvi_lucknow.tif
data\processed\lucknow\fine\predictors\nir_lucknow.tif
data\processed\lucknow\fine\predictors\nir_lucknow_reflectance.tif
data\processed\lucknow\fine\predictors\red_lucknow.tif
data\processed\lucknow\fine\predictors\red

## 7.6 — Recover the Original Model Training Workflow

The trained Random Forest and XGBoost prediction rasters are available from the previous modelling workflow. However, SHAP requires access to the corresponding model structure and the exact predictor matrix used during training.

The original model-training notebooks are therefore inspected to recover:

- predictor variables,
- target variable,
- training data preparation,
- model parameters,
- and preprocessing steps.

The SHAP analysis will reproduce the original modelling workflow rather than creating an unrelated model.

In [6]:
import json

# Paths to the original model-training notebooks
rf_notebook = PROJECT_DIR / "notebooks" / "04_RF_Downscaling.ipynb"
xgb_notebook = PROJECT_DIR / "notebooks" / "05_XGB_Downscaling.ipynb"

# Load the notebooks
with open(rf_notebook, "r", encoding="utf-8") as f:
    rf_nb = json.load(f)

with open(xgb_notebook, "r", encoding="utf-8") as f:
    xgb_nb = json.load(f)

print("RF notebook cells:", len(rf_nb["cells"]))
print("XGB notebook cells:", len(xgb_nb["cells"]))

RF notebook cells: 90
XGB notebook cells: 53


## 7.7 — Locate the Original Model-Training Code

The Random Forest and XGBoost notebooks contain the complete modelling workflows used to generate the existing 30 m LST products.

Rather than manually copying code from those notebooks, the relevant cells are identified programmatically.

The search focuses on the model definitions, predictor construction, target construction, and training/prediction steps.

This ensures that the SHAP analysis remains consistent with the original models.

In [7]:
# Search the original notebooks for important modelling keywords

keywords = [
    "RandomForestRegressor",
    "XGBRegressor",
    "features",
    "feature",
    "predictor",
    "target",
    "X =",
    "y =",
    "fit(",
    "predict("
]


def find_cells(notebook, keywords):
    results = []

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            results.append({
                "cell": i,
                "keywords": matched,
                "source": source
            })

    return results


rf_matches = find_cells(rf_nb, keywords)
xgb_matches = find_cells(xgb_nb, keywords)

print("RF matching cells:", len(rf_matches))
print("XGB matching cells:", len(xgb_matches))

RF matching cells: 28
XGB matching cells: 17


## 7.8 — Extract the Core Model-Training Cells

The previous search identified cells containing general modelling keywords. Many of those cells may only perform plotting, validation, or prediction.

The next step isolates the cells most relevant to reconstructing the original machine-learning workflow:

- predictor/feature construction,
- target construction,
- Random Forest model definition,
- XGBoost model definition,
- model fitting,
- and prediction.

Only these cells are needed to establish the exact model inputs and configuration before applying SHAP.

In [8]:
# Search for the most important model-training keywords

core_keywords = [
    "RandomForestRegressor",
    "XGBRegressor",
    "fit(",
    "X =",
    "y =",
    "feature_cols",
    "features =",
    "predictor_cols",
    "predictors =",
    "target ="
]


def find_core_cells(notebook, keywords):
    results = []

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            results.append((i, matched, source))

    return results


rf_core = find_core_cells(rf_nb, core_keywords)
xgb_core = find_core_cells(xgb_nb, core_keywords)

print("RF core cells:")
for cell_num, matched, _ in rf_core:
    print(f"  Cell {cell_num}: {matched}")

print("\nXGB core cells:")
for cell_num, matched, _ in xgb_core:
    print(f"  Cell {cell_num}: {matched}")

RF core cells:
  Cell 14: ['X =']
  Cell 20: ['X =', 'y =']
  Cell 27: ['X =']
  Cell 28: ['X =', 'y =']
  Cell 47: ['RandomForestRegressor']
  Cell 48: ['RandomForestRegressor']
  Cell 49: ['fit(']
  Cell 62: ['X =']
  Cell 64: ['y =']

XGB core cells:
  Cell 1: ['XGBRegressor']
  Cell 6: ['X =']
  Cell 9: ['X =', 'y =']
  Cell 16: ['XGBRegressor']
  Cell 17: ['fit(']
  Cell 34: ['predictors =']


## 7.9 — Inspect the Original Model Construction

The identified cells are now inspected directly to determine which feature matrix was used for model training.

Because the notebooks contain multiple intermediate `X` and `y` assignments, the surrounding code must be examined before selecting the correct training dataset.

The Random Forest and XGBoost model definitions will also be inspected to recover their original parameters.

In [9]:
# Print selected cells from the original notebooks

rf_cells_to_inspect = [14, 20, 27, 28, 47, 48, 49, 62, 64]
xgb_cells_to_inspect = [1, 6, 9, 16, 17, 34]


def print_selected_cells(notebook, cell_numbers, name):
    print("=" * 80)
    print(name)
    print("=" * 80)

    for cell_num in cell_numbers:
        cell = notebook["cells"][cell_num]

        print(f"\n{'-' * 80}")
        print(f"CELL {cell_num}")
        print(f"{'-' * 80}")

        source = "".join(cell.get("source", []))
        print(source)


print_selected_cells(
    rf_nb,
    rf_cells_to_inspect,
    "RANDOM FOREST NOTEBOOK"
)

print_selected_cells(
    xgb_nb,
    xgb_cells_to_inspect,
    "XGBOOST NOTEBOOK"
)

RANDOM FOREST NOTEBOOK

--------------------------------------------------------------------------------
CELL 14
--------------------------------------------------------------------------------
predictors_90m = {}

for name, path in predictor_paths.items():

    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)

    predictors_90m[name] = aggregate_30m_to_90m(data)

    print(
        f"{name.upper():6} | "
        f"shape = {predictors_90m[name].shape} | "
        f"min = {np.nanmin(predictors_90m[name]):.6f} | "
        f"max = {np.nanmax(predictors_90m[name]):.6f} | "
        f"mean = {np.nanmean(predictors_90m[name]):.6f}"
    )

--------------------------------------------------------------------------------
CELL 20
--------------------------------------------------------------------------------
X = np.column_stack([
    predictors_90m["green"][common_mask],
    predictors_90m["swir1"][common_mask],
    predictors_90m["swir2"][common_mask],
    predict

## 7.10 — Recover the Original Train/Test Split

The original Random Forest and XGBoost models were trained using `X_train` and `y_train`.

Before reconstructing the models for SHAP analysis, the exact procedure used to create the training and testing datasets must be verified.

This is important because the trained model depends on the observations and preprocessing used during training.

The same data-splitting procedure will be reproduced in Notebook 7 so that the SHAP analysis corresponds to the original modelling workflow.

In [10]:
# Search the original notebooks for train/test splitting

split_keywords = [
    "train_test_split",
    "X_train",
    "X_test",
    "y_train",
    "y_test"
]


def find_split_cells(notebook, keywords):
    results = []

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if any(keyword.lower() in source.lower() for keyword in keywords):
            results.append((i, source))

    return results


rf_split = find_split_cells(rf_nb, split_keywords)
xgb_split = find_split_cells(xgb_nb, split_keywords)

print("=" * 80)
print("RANDOM FOREST — TRAIN/TEST SPLIT CELLS")
print("=" * 80)

for cell_num, source in rf_split:
    print(f"\n--- CELL {cell_num} ---")
    print(source)

print("\n" + "=" * 80)
print("XGBOOST — TRAIN/TEST SPLIT CELLS")
print("=" * 80)

for cell_num, source in xgb_split:
    print(f"\n--- CELL {cell_num} ---")
    print(source)

RANDOM FOREST — TRAIN/TEST SPLIT CELLS

--- CELL 41 ---
X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

--- CELL 49 ---
rf_model.fit(X_train, y_train)

--- CELL 50 ---
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

--- CELL 52 ---
train_mae, train_rmse, train_r2 = calculate_metrics(
    y_train,
    y_train_pred
)

test_mae, test_rmse, test_r2 = calculate_metrics(
    y_test,
    y_test_pred
)

--- CELL 54 ---
plt.figure(figsize=(7, 7))

plt.scatter(
    y_test,
    y_test_pred,
    s=10,
    alpha=0.4
)

min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Observed 90 m LST (°C)")
plt.ylabel("Predicted 90 m LST (°C)")
plt.title("Random Forest: Ob

## 7.11 — Recover the Original Training and Testing Masks

The Random Forest and XGBoost workflows use `train_mask` and `test_mask` to divide the valid 90 m pixels into training and testing observations.

Unlike a conventional `train_test_split()` call, these masks may preserve a specific spatial sampling or spatial partitioning strategy.

Because the same training observations must be used to reproduce the original models, the construction of these masks must be identified before rebuilding the models for SHAP analysis.

In [11]:
# Search specifically for the construction of train_mask and test_mask

mask_keywords = [
    "train_mask =",
    "test_mask =",
    "train_mask",
    "test_mask"
]


def find_mask_cells(notebook):
    results = []

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if "train_mask" in source or "test_mask" in source:
            results.append((i, source))

    return results


rf_masks = find_mask_cells(rf_nb)
xgb_masks = find_mask_cells(xgb_nb)

print("=" * 80)
print("RF — MASK CONSTRUCTION")
print("=" * 80)

for cell_num, source in rf_masks:
    print(f"\n--- CELL {cell_num} ---")
    print(source)

print("\n" + "=" * 80)
print("XGB — MASK CONSTRUCTION")
print("=" * 80)

for cell_num, source in xgb_masks:
    print(f"\n--- CELL {cell_num} ---")
    print(source)

RF — MASK CONSTRUCTION

--- CELL 40 ---
train_mask = np.isin(block_ids, train_blocks)
test_mask = np.isin(block_ids, test_blocks)

print("Training observations:", train_mask.sum())
print("Testing observations:", test_mask.sum())
print(
    "Training percentage:",
    train_mask.sum() / len(block_ids) * 100
)
print(
    "Testing percentage:",
    test_mask.sum() / len(block_ids) * 100
)

--- CELL 41 ---
X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

--- CELL 43 ---
split_map = np.full(lst_90m.shape, np.nan)

# Training pixels
split_map[
    rows[train_mask],
    cols[train_mask]
] = 1

# Testing pixels
split_map[
    rows[test_mask],
    cols[test_mask]
] = 2

plt.figure(figsize=(10, 8))

plt.imshow(split_map, origin="upper")

cbar = plt.colorbar()
cbar.set_ticks([1, 2])
cbar.set_ticklabels(["Training", "Testing

## 7.12 — Verify Consistency of the Training/Test Split

Random Forest and XGBoost are being compared as alternative machine-learning
models for the same LST downscaling problem.

For a fair comparison, both models should use the same predictor variables,
target variable, valid-pixel mask, and training/testing partition.

The training workflow is therefore checked for consistency between the two
models before reconstructing them for SHAP analysis.

In [13]:
# Check whether RF and XGBoost use the same spatial train/test split

def get_mask_code(notebook):
    mask_code = []

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if "train_mask" in source or "test_mask" in source:
            mask_code.append((i, source))

    return mask_code


rf_mask_code = get_mask_code(rf_nb)
xgb_mask_code = get_mask_code(xgb_nb)

print("=" * 80)
print("RANDOM FOREST")
print("=" * 80)

for cell_num, source in rf_mask_code:
    print(f"\nCell {cell_num}:")
    print(source)

print("\n" + "=" * 80)
print("XGBOOST")
print("=" * 80)

for cell_num, source in xgb_mask_code:
    print(f"\nCell {cell_num}:")
    print(source)

RANDOM FOREST

Cell 40:
train_mask = np.isin(block_ids, train_blocks)
test_mask = np.isin(block_ids, test_blocks)

print("Training observations:", train_mask.sum())
print("Testing observations:", test_mask.sum())
print(
    "Training percentage:",
    train_mask.sum() / len(block_ids) * 100
)
print(
    "Testing percentage:",
    test_mask.sum() / len(block_ids) * 100
)

Cell 41:
X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Cell 43:
split_map = np.full(lst_90m.shape, np.nan)

# Training pixels
split_map[
    rows[train_mask],
    cols[train_mask]
] = 1

# Testing pixels
split_map[
    rows[test_mask],
    cols[test_mask]
] = 2

plt.figure(figsize=(10, 8))

plt.imshow(split_map, origin="upper")

cbar = plt.colorbar()
cbar.set_ticks([1, 2])
cbar.set_ticklabels(["Training", "Testing"])

plt.title("Spatial Train-

### Verification Result

Random Forest and XGBoost use the same spatial train-test partition.

Both models construct the training and testing masks using the same `block_ids`,
`train_blocks`, and `test_blocks` framework:

```python
train_mask = np.isin(block_ids, train_blocks)
test_mask = np.isin(block_ids, test_blocks)

## 7.13 — Verify the Spatial Train-Test Partition

The models use a block-based spatial partition rather than randomly assigning
individual pixels to the training and testing datasets.

The partition is verified by checking:

- the number of unique spatial blocks,
- the number of training observations,
- the number of testing observations,
- overlap between training and testing masks,
- overlap between training and testing blocks,
- and the proportion of observations assigned to each set.

This confirms that the spatial partition is valid before reconstructing the
models for SHAP analysis.

### 7.13A — Reconstruct the Spatial Block Definitions

The spatial train-test masks depend on three variables:

- `block_ids`
- `train_blocks`
- `test_blocks`

These variables were created in the original Random Forest and XGBoost workflows but
are not automatically available in the new SHAP notebook.

Before verifying the partition, their original construction must therefore be
reproduced.

The original code is inspected first so that the SHAP notebook uses the same
spatial partition as the modelling notebooks.

In [15]:
# Find the cells where the spatial block variables are CREATED

block_creation_keywords = [
    "block_ids =",
    "train_blocks =",
    "test_blocks ="
]

def find_block_creation_cells(notebook):
    results = []

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if any(keyword in source for keyword in block_creation_keywords):
            results.append((i, source))

    return results


rf_block_creation = find_block_creation_cells(rf_nb)
xgb_block_creation = find_block_creation_cells(xgb_nb)

print("=" * 80)
print("RF — BLOCK CREATION")
print("=" * 80)

for cell_num, source in rf_block_creation:
    print(f"\n--- CELL {cell_num} ---")
    print(source)

print("\n" + "=" * 80)
print("XGB — BLOCK CREATION")
print("=" * 80)

for cell_num, source in xgb_block_creation:
    print(f"\n--- CELL {cell_num} ---")
    print(source)

RF — BLOCK CREATION

--- CELL 31 ---
BLOCK_SIZE = 30

block_rows = rows // BLOCK_SIZE
block_cols = cols // BLOCK_SIZE

block_ids = (
    block_rows * (254 // BLOCK_SIZE + 1)
    + block_cols
)

print("Number of unique spatial blocks:", np.unique(block_ids).size)

--- CELL 38 ---
eligible_test_blocks = block_counts[
    block_counts >= MIN_TEST_PIXELS
].index.to_numpy()

print("Total spatial blocks:", len(block_counts))
print("Eligible test blocks:", len(eligible_test_blocks))
print("Excluded from test selection:", 
      len(block_counts) - len(eligible_test_blocks))

--- CELL 39 ---
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

test_blocks = rng.choice(
    eligible_test_blocks,
    size=11,
    replace=False
)

train_blocks = np.array([
    block for block in block_counts.index
    if block not in test_blocks
])

print("Training blocks:", len(train_blocks))
print("Testing blocks:", len(test_blocks))

print("\nTest block IDs:")
print(np.sort(test_blocks))

XGB — BLOCK 

## 7.13B — Reconstruct the Spatial Block Selection

The previous modelling workflow divides the 90 m LST grid into spatial blocks before
creating the training and testing datasets.

First, a spatial block ID is assigned to each pixel using a fixed block size. The
resulting `block_ids` identify which spatial block each observation belongs to.

The original workflow then uses these block IDs to select testing blocks and
construct the corresponding training blocks.

This step inspects and reproduces the original block-selection procedure so that
Notebook 7 uses the same spatial partition as the Random Forest and XGBoost
modelling notebooks.

The spatial partition is important because observations from nearby pixels can be
spatially correlated. Keeping entire blocks together helps evaluate whether the
model can generalize to spatial areas that were not used during training.

In [16]:
# Print the complete cells responsible for selecting
# testing and training spatial blocks

important_cells = [31, 32, 33, 34, 35, 36, 37, 38, 39]

print("=" * 90)
print("SPATIAL BLOCK SELECTION — RF NOTEBOOK")
print("=" * 90)

for cell_num in important_cells:

    if cell_num >= len(rf_nb["cells"]):
        continue

    cell = rf_nb["cells"][cell_num]

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    print(f"\n{'-' * 90}")
    print(f"CELL {cell_num}")
    print(f"{'-' * 90}")
    print(source)

SPATIAL BLOCK SELECTION — RF NOTEBOOK

------------------------------------------------------------------------------------------
CELL 31
------------------------------------------------------------------------------------------
BLOCK_SIZE = 30

block_rows = rows // BLOCK_SIZE
block_cols = cols // BLOCK_SIZE

block_ids = (
    block_rows * (254 // BLOCK_SIZE + 1)
    + block_cols
)

print("Number of unique spatial blocks:", np.unique(block_ids).size)

------------------------------------------------------------------------------------------
CELL 32
------------------------------------------------------------------------------------------
# Create a 2D map of spatial block IDs
block_map = np.full(lst_90m.shape, np.nan)

block_map[common_mask] = block_ids

plt.figure(figsize=(10, 8))
plt.imshow(block_map, origin="upper")
plt.colorbar(label="Spatial Block ID")
plt.title("Spatial Blocks for Random Forest Evaluation")
plt.xlabel("Column")
plt.ylabel("Row")
plt.show()

----------------------

## 7.13C — Inspect the Test-Block Selection Logic

The spatial block IDs have been created, but the complete procedure used to
select the testing blocks has not yet been recovered.

The following cells are inspected individually to identify:

- how the number of valid pixels in each block is calculated,
- the minimum number of pixels required for a test block,
- how testing blocks are selected,
- how training blocks are defined,
- and whether a fixed random seed is used.

The original procedure will be reproduced exactly in Notebook 7 rather than
creating a new spatial partition.

In [17]:
# Print cells 33–39 individually so the output is not truncated

for cell_num in range(33, 40):

    cell = rf_nb["cells"][cell_num]

    print("\n" + "=" * 90)
    print(f"CELL {cell_num}")
    print("=" * 90)

    if cell["cell_type"] == "code":
        print("".join(cell.get("source", [])))
    else:
        print("[Markdown cell]")
        print("".join(cell.get("source", [])))


CELL 33
block_counts = pd.Series(block_ids).value_counts().sort_index()

print(block_counts)
print("\nNumber of blocks:", len(block_counts))
print("Minimum pixels per block:", block_counts.min())
print("Maximum pixels per block:", block_counts.max())
print("Mean pixels per block:", block_counts.mean())

CELL 34
print("Smallest blocks:")
print(block_counts.sort_values().head(10))

print("\nLargest blocks:")
print(block_counts.sort_values(ascending=False).head(10))

CELL 35
[Markdown cell]
### Observation

After applying the common validity mask, 35,383 valid 90 m observations were retained from 59,436 total pixels, corresponding to 59.53% of the coarse grid.

The final ML dataset contains six predictors: Green, SWIR1, SWIR2, NDVI, NDBI, and MNDWI.

No NaN or infinite values remain in the final dataset, ensuring that the Random Forest receives a fully valid training matrix.

CELL 36
[Markdown cell]
### Spatial Train-Test Split

Because neighboring pixels are spatially dependent, a rando

## 7.14 — Reconstruct the Original Spatial Train-Test Split

The original modelling workflow divides the valid 90 m observations into spatial
blocks of 30 × 30 pixels, corresponding to approximately 2.7 km × 2.7 km at
the 90 m resolution.

Blocks containing fewer than 100 valid observations are excluded from test-block
selection. Eleven eligible blocks are then randomly selected for testing using
a fixed random seed of 42.

All remaining blocks are assigned to the training set.

This procedure is reproduced exactly in the SHAP notebook so that any model
reconstructed here uses the same spatial partition as the original Random Forest
and XGBoost models.

## 7.14A — Establish the 90 m Spatial Grid

The spatial block IDs in the original modelling workflow are calculated from
the row and column positions of the 90 m LST grid.

Notebook 7 is a separate execution environment, so variables such as `rows`
and `cols` from the original modelling notebook are not automatically
available.

The 90 m LST raster is therefore loaded first so that the spatial dimensions
used to construct the original block IDs can be recovered directly from the
project data.

In [19]:
import rasterio

# Path to the 90 m coarse LST raster
lst_90m_path = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "lucknow"
    / "coarse"
    / "lst_90m_lucknow.tif"
)

# Load the 90 m LST raster
with rasterio.open(lst_90m_path) as src:
    lst_90m = src.read(1).astype(np.float32)
    lst_profile = src.profile.copy()
    lst_transform = src.transform
    lst_crs = src.crs

# Get raster dimensions
rows, cols = lst_90m.shape

print("90 m LST shape:", lst_90m.shape)
print("Rows:", rows)
print("Columns:", cols)
print("CRS:", lst_crs)

90 m LST shape: (234, 254)
Rows: 234
Columns: 254
CRS: EPSG:32644


## 7.14B — Load the 30 m Predictor Rasters

The Random Forest and XGBoost models use six predictor variables:

- Green reflectance
- SWIR1 reflectance
- SWIR2 reflectance
- NDVI
- NDBI
- MNDWI

These predictors are available as 30 m raster layers.

The original modelling workflow aggregates these 30 m predictors to the 90 m
coarse LST grid before constructing the machine-learning training matrix.

The same predictor set and spatial resolution will therefore be reconstructed
in Notebook 7.

In [20]:
# Define the six predictor raster paths

predictor_dir = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "lucknow"
    / "fine"
    / "predictors"
)

predictor_paths = {
    "green": predictor_dir / "green_lucknow_reflectance.tif",
    "swir1": predictor_dir / "swir1_lucknow_reflectance.tif",
    "swir2": predictor_dir / "swir2_lucknow_reflectance.tif",
    "ndvi": predictor_dir / "ndvi_lucknow.tif",
    "ndbi": predictor_dir / "ndbi_lucknow.tif",
    "mndwi": predictor_dir / "mndwi_lucknow.tif"
}

# Check that all required files exist

for name, path in predictor_paths.items():
    print(f"{name:6} -> {path.exists()} -> {path.name}")

green  -> True -> green_lucknow_reflectance.tif
swir1  -> True -> swir1_lucknow_reflectance.tif
swir2  -> True -> swir2_lucknow_reflectance.tif
ndvi   -> True -> ndvi_lucknow.tif
ndbi   -> True -> ndbi_lucknow.tif
mndwi  -> True -> mndwi_lucknow.tif


## 7.14C — Recover the 30 m to 90 m Aggregation Function

The original modelling notebooks aggregate the 30 m predictor rasters to the
90 m coarse grid using the `aggregate_30m_to_90m` function.

Because Notebook 7 is a separate notebook, this function is not automatically
available in its Python session.

The original function definition is therefore recovered from the Random Forest
workflow so that the predictor aggregation used for SHAP reconstruction is
identical to the original modelling workflow.

In [21]:
# Find the definition of aggregate_30m_to_90m in the RF notebook

for i, cell in enumerate(rf_nb["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if "def aggregate_30m_to_90m" in source:
        print("=" * 80)
        print(f"FUNCTION DEFINITION FOUND IN RF CELL {i}")
        print("=" * 80)
        print(source)

FUNCTION DEFINITION FOUND IN RF CELL 13
def aggregate_30m_to_90m(data, nodata=None):
    """
    Aggregate a 30 m raster to 90 m using 3x3 valid-pixel means.

    The input must have dimensions compatible with a 3x3 aggregation.
    """

    # Remove the incomplete final row
    data = data[:702, :762]

    # Reshape into 3x3 blocks
    reshaped = data.reshape(234, 3, 254, 3)

    # Treat non-finite values as invalid
    valid = np.isfinite(reshaped)

    # Sum only valid values
    data_clean = np.where(valid, reshaped, 0)

    value_sum = data_clean.sum(axis=(1, 3))
    valid_count = valid.sum(axis=(1, 3))

    # Calculate mean only where at least one valid pixel exists
    result = np.full((234, 254), np.nan, dtype=np.float32)

    valid_blocks = valid_count > 0

    result[valid_blocks] = (
        value_sum[valid_blocks] /
        valid_count[valid_blocks]
    )

    return result
FUNCTION DEFINITION FOUND IN RF CELL 25
def aggregate_30m_to_90m(data, nodata=None):
    """
    Aggr

## 7.14D — Recover the Complete 30 m to 90 m Aggregation Function

The original Random Forest workflow uses a custom function called
`aggregate_30m_to_90m()` to convert the 30 m predictor rasters to the 90 m
coarse grid.

The function performs 3 × 3 aggregation using valid-pixel means and handles
invalid or missing raster values.

Because the function is not automatically available in Notebook 7, its
complete definition is recovered from the original modelling notebook before
being reproduced here.

Using the original function is important because changing the aggregation
procedure could change the predictor values and therefore produce a different
machine-learning dataset from the one used to train the original models.

In [22]:
# Extract the complete aggregation function source

aggregation_cell = rf_nb["cells"][13]

aggregation_source = "".join(
    aggregation_cell.get("source", [])
)

print(aggregation_source)

def aggregate_30m_to_90m(data, nodata=None):
    """
    Aggregate a 30 m raster to 90 m using 3x3 valid-pixel means.

    The input must have dimensions compatible with a 3x3 aggregation.
    """

    # Remove the incomplete final row
    data = data[:702, :762]

    # Reshape into 3x3 blocks
    reshaped = data.reshape(234, 3, 254, 3)

    # Treat non-finite values as invalid
    valid = np.isfinite(reshaped)

    # Sum only valid values
    data_clean = np.where(valid, reshaped, 0)

    value_sum = data_clean.sum(axis=(1, 3))
    valid_count = valid.sum(axis=(1, 3))

    # Calculate mean only where at least one valid pixel exists
    result = np.full((234, 254), np.nan, dtype=np.float32)

    valid_blocks = valid_count > 0

    result[valid_blocks] = (
        value_sum[valid_blocks] /
        valid_count[valid_blocks]
    )

    return result


## 7.14E — Recreate the 30 m to 90 m Aggregation Function

The original modelling workflow aggregates each 30 m predictor raster to the
90 m coarse LST grid using a 3 × 3 valid-pixel mean.

Each 90 m cell corresponds to a 3 × 3 group of 30 m pixels. Only finite
(valid) pixels are included in the mean. If all nine input pixels are invalid,
the corresponding 90 m output cell is assigned `NaN`.

The same aggregation function is reproduced here to ensure that the predictor
values used for SHAP reconstruction are consistent with the original
Random Forest and XGBoost training workflow.

In [23]:
def aggregate_30m_to_90m(data, nodata=None):
    """
    Aggregate a 30 m raster to 90 m using 3x3 valid-pixel means.

    The input must have dimensions compatible with a 3x3 aggregation.
    """

    # Remove the incomplete final row and columns
    data = data[:702, :762]

    # Reshape into 3x3 blocks
    reshaped = data.reshape(234, 3, 254, 3)

    # Treat non-finite values as invalid
    valid = np.isfinite(reshaped)

    # Sum only valid values
    data_clean = np.where(valid, reshaped, 0)

    value_sum = data_clean.sum(axis=(1, 3))
    valid_count = valid.sum(axis=(1, 3))

    # Calculate mean only where at least one valid pixel exists
    result = np.full((234, 254), np.nan, dtype=np.float32)

    valid_blocks = valid_count > 0

    result[valid_blocks] = (
        value_sum[valid_blocks]
        / valid_count[valid_blocks]
    )

    return result

## 7.14F — Validate the Aggregation Function

Before applying the aggregation function to all predictor variables, it is
tested on the Green reflectance raster.

The resulting 90 m raster should have the same spatial dimensions as the
90 m coarse LST raster.

This validation confirms that the 30 m to 90 m aggregation is functioning
correctly before the complete predictor matrix is constructed.

In [24]:
# Test the aggregation function using the Green predictor

green_path = predictor_paths["green"]

with rasterio.open(green_path) as src:
    green_30m = src.read(1).astype(np.float32)

print("Original Green raster shape:", green_30m.shape)

green_90m = aggregate_30m_to_90m(green_30m)

print("Aggregated Green raster shape:", green_90m.shape)
print("Expected 90 m LST shape:", lst_90m.shape)

print("\nGreen 90 m statistics:")
print("Min :", np.nanmin(green_90m))
print("Max :", np.nanmax(green_90m))
print("Mean:", np.nanmean(green_90m))

Original Green raster shape: (703, 762)
Aggregated Green raster shape: (234, 254)
Expected 90 m LST shape: (234, 254)

Green 90 m statistics:
Min : 0.0
Max : 17696.111
Mean: 7000.8955


## 7.14G — Verify Predictor Scaling

The Green predictor has the expected spatial dimensions after aggregation, but
its numerical range must also be verified.

The aggregated Green values currently range from 0 to approximately 17,696.
This is substantially different from a typical reflectance representation.

Before constructing the SHAP training matrix, the original modelling workflow
must therefore be checked to determine whether:

- the Green raster contains scaled digital values,
- a scaling operation is applied before model training,
- or the predictor file is intentionally stored in another numerical format.

The same numerical representation used by the original Random Forest and
XGBoost models must be preserved in the SHAP reconstruction.

In [25]:
# Find where predictor_paths is defined in the original RF notebook

for i, cell in enumerate(rf_nb["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if "predictor_paths" in source:
        print("=" * 80)
        print(f"RF CELL {i}")
        print("=" * 80)
        print(source)

RF CELL 4
predictor_paths = {
    "green": FINE_DIR / "predictors"/ "green_lucknow_reflectance.tif",
    "swir1": FINE_DIR / "predictors"/ "swir1_lucknow_reflectance.tif",
    "swir2": FINE_DIR / "predictors"/ "swir2_lucknow_reflectance.tif",
    "ndvi": FINE_DIR / "predictors"/ "ndvi_lucknow.tif",
    "ndbi": FINE_DIR / "predictors"/ "ndbi_lucknow.tif",
    "mndwi": FINE_DIR / "predictors"/ "mndwi_lucknow.tif"
}

lst_90m_path = COARSE_DIR / "lst_90m_lucknow.tif"
RF CELL 5
for name, path in predictor_paths.items():
    print(f"{name:6}:", path.exists(), "->", path.name)

print(f"{'lst_90m':6}:", lst_90m_path.exists(), "->", lst_90m_path.name)
RF CELL 7
for name, path in predictor_paths.items():
    print(f"\n{name.upper()}")
    print(raster_info(path))

print("\nLST 90M")
print(raster_info(lst_90m_path))
RF CELL 8
for name in ["green", "swir1", "swir2"]:
    path = predictor_paths[name]

    with rasterio.open(path) as src:
        data = src.read(1)

        valid = data[data != src.

## 7.14G — Trace Predictor Preprocessing and Scaling

The Random Forest notebook loads the predictor rasters directly from the processed
predictor directory.

The aggregated Green raster currently has values ranging from 0 to approximately
17,696. Before constructing the machine-learning feature matrix, the preprocessing
workflow used to create the predictor rasters must therefore be verified.

This step traces the Green, SWIR1, and SWIR2 predictor preparation back to the
predictor-generation workflow.

The purpose is to determine whether the stored values are:

- raw digital values,
- scaled reflectance values,
- or another processed representation.

No additional scaling will be introduced unless it is present in the original
project workflow.

In [26]:
# Load the predictor-generation notebook

predictor_notebook_path = (
    PROJECT_DIR
    / "notebooks"
    / "02_Predictor_Generation.ipynb"
)

with open(predictor_notebook_path, "r", encoding="utf-8") as f:
    predictor_nb = json.load(f)

print(
    "Predictor-generation notebook cells:",
    len(predictor_nb["cells"])
)

Predictor-generation notebook cells: 46


In [27]:
# Find where the reflectance predictor files are created or scaled

scaling_keywords = [
    "green_lucknow_reflectance",
    "swir1_lucknow_reflectance",
    "swir2_lucknow_reflectance",
    "reflectance",
    "scale",
    "0.0001",
    "10000"
]

for i, cell in enumerate(predictor_nb["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(
        keyword.lower() in source.lower()
        for keyword in scaling_keywords
    ):
        print("\n" + "=" * 90)
        print(f"CELL {i}")
        print("=" * 90)
        print(source)


CELL 10
# Landsat Collection 2 Surface Reflectance scaling

SCALE_FACTOR = 0.0000275
OFFSET = -0.2

# Convert stored values to reflectance
green_reflectance = green.astype("float32") * SCALE_FACTOR + OFFSET
swir1_reflectance = swir1.astype("float32") * SCALE_FACTOR + OFFSET
swir2_reflectance = swir2.astype("float32") * SCALE_FACTOR + OFFSET

# Remove NoData pixels
green_reflectance[green == 0] = np.nan
swir1_reflectance[swir1 == 0] = np.nan
swir2_reflectance[swir2 == 0] = np.nan

print("Green reflectance range:",
      np.nanmin(green_reflectance),
      "to",
      np.nanmax(green_reflectance))

print("SWIR1 reflectance range:",
      np.nanmin(swir1_reflectance),
      "to",
      np.nanmax(swir1_reflectance))

print("SWIR2 reflectance range:",
      np.nanmin(swir2_reflectance),
      "to",
      np.nanmax(swir2_reflectance))

CELL 18
# Clip the three bands to the Lucknow boundary

green_lucknow_reflectance, green_profile = clip_raster_to_boundary(
    green_path,
    lucknow_bound

## 7.14H — Verify the Saved Predictor Raster Values

The predictor-generation workflow applies the Landsat Collection 2 Surface
Reflectance scaling:

\[
\rho = DN \times 0.0000275 - 0.2
\]

where \(DN\) represents the stored Landsat value and \(\rho\) represents
surface reflectance.

The saved predictor files are then used directly by the Random Forest and
XGBoost workflows.

Because the aggregated Green raster produced unexpectedly large values, the
actual numerical values stored in the saved predictor GeoTIFF are verified
before continuing.

This check determines whether the saved predictor files contain the scaled
reflectance values generated by the preprocessing workflow.

In [28]:
# Inspect the actual stored values in the six predictor GeoTIFFs

for name, path in predictor_paths.items():

    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)

    finite = np.isfinite(data)

    print(f"\n{name.upper()}")
    print("-" * 40)
    print("Shape :", data.shape)
    print("Min   :", np.nanmin(data[finite]))
    print("Max   :", np.nanmax(data[finite]))
    print("Mean  :", np.nanmean(data[finite]))


GREEN
----------------------------------------
Shape : (703, 762)
Min   : 0.0
Max   : 22311.0
Mean  : 6990.9375

SWIR1
----------------------------------------
Shape : (703, 762)
Min   : 0.0
Max   : 44110.0
Mean  : 9371.458

SWIR2
----------------------------------------
Shape : (703, 762)
Min   : 0.0
Max   : 36440.0
Mean  : 8107.6875

NDVI
----------------------------------------
Shape : (703, 762)
Min   : 0.027113719
Max   : 0.77813345
Mean  : 0.30725604

NDBI
----------------------------------------
Shape : (703, 762)
Min   : -0.52347004
Max   : 0.013186339
Mean  : -0.2225873

MNDWI
----------------------------------------
Shape : (703, 762)
Min   : -0.3281944
Max   : 0.06609564
Mean  : -0.14413378


## 7.14I — Verify the Saved Reflectance Outputs

The predictor-generation workflow calculates surface reflectance from the
Landsat Collection 2 stored values using a scale factor and additive offset.

However, inspection of the predictor GeoTIFFs shows that the Green, SWIR1, and
SWIR2 files contain values in the thousands, which suggests that the stored
Landsat values may have been written to the output files instead of the
scaled reflectance arrays.

Before reconstructing the machine-learning dataset, the code responsible for
writing these GeoTIFFs is inspected.

This step determines whether the saved files contain the same numerical
representation that was used during the original Random Forest and XGBoost
training workflow.

No additional scaling is applied at this stage.

In [29]:
# Find the code that writes the Green/SWIR reflectance GeoTIFFs

write_keywords = [
    "green_reflectance",
    "swir1_reflectance",
    "swir2_reflectance",
    "green_lucknow_reflectance",
    "swir1_lucknow_reflectance",
    "swir2_lucknow_reflectance",
    "rasterio.open",
    "write("
]

for i, cell in enumerate(predictor_nb["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if (
        "green_lucknow_reflectance" in source
        or "swir1_lucknow_reflectance" in source
        or "swir2_lucknow_reflectance" in source
    ):
        print("\n" + "=" * 90)
        print(f"CELL {i}")
        print("=" * 90)
        print(source)


CELL 18
# Clip the three bands to the Lucknow boundary

green_lucknow_reflectance, green_profile = clip_raster_to_boundary(
    green_path,
    lucknow_boundary_utm
)

swir1_lucknow_reflectance, swir1_profile = clip_raster_to_boundary(
    swir1_path,
    lucknow_boundary_utm
)

swir2_lucknow_reflectance, swir2_profile = clip_raster_to_boundary(
    swir2_path,
    lucknow_boundary_utm
)

print("Green Lucknow shape :", green_lucknow_reflectance.shape)
print("SWIR1 Lucknow shape :", swir1_lucknow_reflectance.shape)
print("SWIR2 Lucknow shape :", swir2_lucknow_reflectance.shape)

CELL 20
plt.figure(figsize=(10, 8))
plt.imshow(green_lucknow_reflectance, cmap="Greens")
plt.title("Green Band - Lucknow")
plt.show()

CELL 21
plt.figure(figsize=(10, 8))
plt.imshow(swir1_lucknow_reflectance, cmap="Purples")
plt.title("SWIR1 Band - Lucknow")
plt.show()

CELL 22
plt.figure(figsize=(10, 8))
plt.imshow(swir2_lucknow_reflectance, cmap="Oranges")
plt.title("SWIR2 Band - Lucknow")
plt.show()

CELL 24

## 7.14J — Identify the Predictor GeoTIFF Writing Step

The predictor-generation workflow first calculates surface reflectance and then
clips the reflectance arrays to the Lucknow boundary.

The clipped arrays are subsequently saved as GeoTIFF files that are used by the
machine-learning workflow.

The exact raster-writing step is inspected to determine which array is written
to the Green, SWIR1, and SWIR2 predictor files.

This verification is necessary because the numerical values currently stored in
the predictor files are substantially larger than typical surface-reflectance
values.

In [30]:
# Find cells that write raster data in the predictor-generation notebook

for i, cell in enumerate(predictor_nb["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if (
        ".write(" in source
        or "green_lucknow_reflectance.tif" in source
        or "swir1_lucknow_reflectance.tif" in source
        or "swir2_lucknow_reflectance.tif" in source
        or "green_profile" in source
        or "swir1_profile" in source
        or "swir2_profile" in source
    ):
        print("\n" + "=" * 90)
        print(f"CELL {i}")
        print("=" * 90)
        print(source)


CELL 7
# File paths for the required bands

green_path = LANDSAT_DIR / "LC08_L2SP_144041_20260519_20260528_02_T1_SR_B3.TIF"
swir1_path = LANDSAT_DIR / "LC08_L2SP_144041_20260519_20260528_02_T1_SR_B6.TIF"
swir2_path = LANDSAT_DIR / "LC08_L2SP_144041_20260519_20260528_02_T1_SR_B7.TIF"

# Open the rasters
with rasterio.open(green_path) as src:
    green = src.read(1)
    green_profile = src.profile
    green_transform = src.transform
    green_crs = src.crs

with rasterio.open(swir1_path) as src:
    swir1 = src.read(1)

with rasterio.open(swir2_path) as src:
    swir2 = src.read(1)

print("Green shape :", green.shape)
print("SWIR1 shape :", swir1.shape)
print("SWIR2 shape :", swir2.shape)
print("CRS         :", green_crs)

CELL 18
# Clip the three bands to the Lucknow boundary

green_lucknow_reflectance, green_profile = clip_raster_to_boundary(
    green_path,
    lucknow_boundary_utm
)

swir1_lucknow_reflectance, swir1_profile = clip_raster_to_boundary(
    swir1_path,
    lucknow_boun

## 7.15 — Aggregate the Six Predictors to the 90 m Grid

The original Random Forest and XGBoost models were trained using six predictor
variables aggregated from 30 m to the 90 m coarse LST grid:

- Green
- SWIR1
- SWIR2
- NDVI
- NDBI
- MNDWI

The same aggregation function recovered from the original modelling workflow
is applied to each predictor.

This produces six 90 m predictor arrays with the same spatial dimensions as
the 90 m LST target.

In [31]:
# Aggregate all six predictors from 30 m to 90 m

predictors_90m = {}

for name, path in predictor_paths.items():

    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)

    predictors_90m[name] = aggregate_30m_to_90m(data)

    print(
        f"{name.upper():6} | "
        f"shape = {predictors_90m[name].shape} | "
        f"min = {np.nanmin(predictors_90m[name]):.6f} | "
        f"max = {np.nanmax(predictors_90m[name]):.6f} | "
        f"mean = {np.nanmean(predictors_90m[name]):.6f}"
    )

GREEN  | shape = (234, 254) | min = 0.000000 | max = 17696.111328 | mean = 7000.895508
SWIR1  | shape = (234, 254) | min = 0.000000 | max = 28746.445312 | mean = 9384.808594
SWIR2  | shape = (234, 254) | min = 0.000000 | max = 27468.888672 | mean = 8119.237305
NDVI   | shape = (234, 254) | min = 0.102692 | max = 0.741849 | mean = 0.308028
NDBI   | shape = (234, 254) | min = -0.460101 | max = -0.009748 | mean = -0.222936
MNDWI  | shape = (234, 254) | min = -0.245560 | max = 0.051391 | mean = -0.144368


## 7.16 — Construct the Common Validity Mask

Machine-learning models cannot use observations containing missing or invalid
values.

The original modelling workflow therefore constructs a common validity mask
using the 90 m LST target and all six predictor variables.

An observation is retained only when:

- the LST value is finite and is not equal to `-9999`, and
- every predictor value is finite.

This ensures that the resulting feature matrix contains complete observations
for all six predictors.

In [32]:
# Construct the common validity mask

common_mask = (
    np.isfinite(lst_90m)
    & (lst_90m != -9999)
)

for name, data in predictors_90m.items():
    common_mask &= np.isfinite(data)

print("Total 90 m pixels:", lst_90m.size)
print("Valid observations:", common_mask.sum())
print(
    "Valid percentage:",
    common_mask.sum() / lst_90m.size * 100
)

print(
    "Invalid observations:",
    (~common_mask).sum()
)

Total 90 m pixels: 59436
Valid observations: 35383
Valid percentage: 59.531260515512486
Invalid observations: 24053


## 7.17 — Construct the Machine-Learning Feature Matrix

The six valid 90 m predictors are converted into a two-dimensional feature
matrix for machine-learning analysis.

The feature order is kept identical to the original Random Forest and XGBoost
workflows:

1. Green
2. SWIR1
3. SWIR2
4. NDVI
5. NDBI
6. MNDWI

The target variable is the corresponding 90 m LST value.

Only observations satisfying the common validity mask are included.

In [33]:
# Define the feature order used by the original models

feature_names = [
    "green",
    "swir1",
    "swir2",
    "ndvi",
    "ndbi",
    "mndwi"
]

# Construct X using the exact original feature order

X = np.column_stack([
    predictors_90m[name][common_mask]
    for name in feature_names
])

# Construct the target variable

y = lst_90m[common_mask]

print("Feature names:", feature_names)
print("X shape:", X.shape)
print("y shape:", y.shape)

Feature names: ['green', 'swir1', 'swir2', 'ndvi', 'ndbi', 'mndwi']
X shape: (35383, 6)
y shape: (35383,)


## 7.18 — Validate the Feature Matrix

The reconstructed feature matrix is checked for missing, infinite, or
unexpectedly incomplete values before model reconstruction.

This validation ensures that the matrix used for SHAP analysis contains the
same number of observations and predictors as the original modelling dataset.

In [34]:
# Validate X and y

print("X contains NaN:", np.isnan(X).any())
print("X contains infinity:", np.isinf(X).any())

print("y contains NaN:", np.isnan(y).any())
print("y contains infinity:", np.isinf(y).any())

print("\nTotal X observations:", X.shape[0])
print("Number of features:", X.shape[1])
print("Target observations:", y.shape[0])

print("\nFeature summary:")
print(
    pd.DataFrame(X, columns=feature_names).describe().round(4)
)

X contains NaN: False
X contains infinity: False
y contains NaN: False
y contains infinity: False

Total X observations: 35383
Number of features: 6
Target observations: 35383

Feature summary:
            green       swir1       swir2        ndvi        ndbi       mndwi
count  35383.0000  35383.0000  35383.0000  35383.0000  35383.0000  35383.0000
mean   11757.0938  15760.5469  13635.2539      0.3080     -0.2229     -0.1444
std     1216.7098   1858.5094   1760.1923      0.1115      0.0489      0.0273
min     1108.8889   1215.1111   1068.4445      0.1027     -0.4601     -0.2456
25%    11491.1665  15152.3330  13050.0557      0.2258     -0.2496     -0.1602
50%    12012.4443  15874.4443  13932.7773      0.2860     -0.2179     -0.1393
75%    12336.1113  16511.5557  14470.5557      0.3636     -0.1891     -0.1260
max    17696.1113  28746.4453  27468.8887      0.7418     -0.0097      0.0514
